<a href="https://colab.research.google.com/github/attilaf3/MSc_Onlab_attilafarago/blob/main/UrbanReID_BABGulyas_02_PAT_teszt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Urban Elements Re-ID — PAT-inspired baseline
Pretrained ViT-B/16 with three regional learnable attention queries, global feature and concatenation. No gating, CSL or PSD. This is a simplified PAT-inspired implementation, not a reproduction of the original PAT. Run data preparation first in the same working directory. All paths are relative to that directory.

In [ ]:
!pip install -q keras-hub

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
import keras_hub
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
from keras import layers, ops

SEED = 2026
keras.utils.set_random_seed(SEED)
OUT = Path("outputs")
ROOT = None  # Set manually if automatic detection fails
IMAGE_SIZE = 224
BATCH_SIZE = 16
EMBEDDING_SIZE = 256
USE_AUGMENTATION = False
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 15
PRESET = "vit_base_patch16_224_imagenet21k"

if ROOT is None:
    candidates = [p.parent for p in Path("data").rglob("train.csv")
                  if (p.parent / "image_train").is_dir()
                  and (p.parent / "image_query").is_dir()
                  and not any("external" in x.lower() for x in p.parts)]
    if len(candidates) != 1:
        raise ValueError(f"Set ROOT manually. Candidates: {candidates}")
    ROOT = candidates[0]
ROOT = Path(ROOT)
train = pd.read_csv(OUT / "local_train.csv", dtype={"object_id": str})
val = pd.read_csv(OUT / "local_validation.csv", dtype={"object_id": str})
query = pd.read_csv(OUT / "local_test_query.csv", dtype={"object_id": str})
gallery = pd.read_csv(OUT / "local_test_gallery.csv", dtype={"object_id": str})
NUM_IDS = train["train_label"].nunique()
if set(train.train_label) != set(range(NUM_IDS)):
    raise ValueError("Training labels must be consecutive integers")
if set(train.object_id) & (set(val.object_id) | set(query.object_id) | set(gallery.object_id)) or set(val.object_id) & (set(query.object_id) | set(gallery.object_id)):
    raise ValueError("Object ID leakage")
if set(query.image_path) & set(gallery.image_path) or not set(query.object_id) <= set(gallery.object_id):
    raise ValueError("Invalid query/gallery split")
print(ROOT, NUM_IDS, len(train), len(val), len(query), len(gallery))

## Image pipeline
Aspect-ratio preserving resize and black padding, as in data preparation, but at 224 pixels for the pretrained checkpoint. Keep pixels in [0,1] until the preset-specific image converter. Augmentation is applied only during training.

In [ ]:
def read_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    return tf.image.resize_with_pad(image, IMAGE_SIZE, IMAGE_SIZE)

def dataset(frame, training=False):
    paths = [str(ROOT / p) for p in frame.image_path]
    if training:
        ds = tf.data.Dataset.from_tensor_slices((paths, frame.train_label.to_numpy(np.int32)))
        ds = ds.shuffle(len(frame), seed=SEED)
        ds = ds.map(lambda p, y: (read_image(p), y), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices(paths)
        ds = ds.map(read_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = dataset(train, True)
val_ds = dataset(val)
query_ds = dataset(query)
gallery_ds = dataset(gallery)

# Optional shared module. No import is needed when disabled.
if USE_AUGMENTATION:
    from augmentation import make_augmentation
    augmentation = make_augmentation(SEED)
else:
    augmentation = keras.Sequential([layers.Activation("linear")], name="augmentation")

## Pretrained backbone
Use the matching image converter rather than assuming ImageNet normalization. Inspect the backbone output: this notebook requires the full CLS + patch sequence, not a pooled vector. The preset and API must be available in the installed KerasHub version.

In [ ]:
backbone = keras_hub.models.ViTBackbone.from_preset(PRESET)
converter = keras_hub.models.ViTImageConverter.from_preset(PRESET)
backbone.trainable = False
print(backbone.output_shape)
print(converter.get_config())
# KerasHub image converters expect raw pixel values in [0,255].
# Inspect the converter config before training if using a different preset.
example = converter(tf.zeros((1, IMAGE_SIZE, IMAGE_SIZE, 3)))
tokens = backbone(example, training=False)
if len(tokens.shape) != 3 or tokens.shape[1] != 197:
    raise ValueError(f"Expected 197 tokens, got {tokens.shape}")
print(tokens.shape)

## Regional part attention
Three overlapping horizontal regions (rows 0–5, 4–9, 8–13). Each has its own learned query token and cross-attention. These are spatial regions, not manually labeled semantic parts. Global CLS and three part features are concatenated; no adaptive gating.

In [ ]:
@keras.saving.register_keras_serializable(package="UrbanReID")
class PartAttention(layers.Layer):
    def __init__(self, dim=768, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.attention = layers.MultiHeadAttention(num_heads=12, key_dim=dim // 12)
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def build(self, input_shape):
        self.part_token = self.add_weight(name="part_token", shape=(1, 1, self.dim),
                                          initializer=keras.initializers.RandomNormal(stddev=0.02))
        super().build(input_shape)

    def call(self, patches):
        q = ops.repeat(self.part_token, ops.shape(patches)[0], axis=0)
        result = self.attention(query=q, key=patches, value=patches)
        return ops.squeeze(self.norm(q + result), axis=1)

    def get_config(self):
        return {**super().get_config(), "dim": self.dim}

inputs = keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = augmentation(inputs)
x = converter(x * 255.0)
tokens = backbone(x, training=False)
global_feature = tokens[:, 0, :]
patches = layers.Reshape((14, 14, 768))(tokens[:, 1:, :])
features = [global_feature]
for i, (start, end) in enumerate([(0, 6), (4, 10), (8, 14)]):
    region = layers.Reshape(((end-start)*14, 768))(patches[:, start:end, :, :])
    features.append(PartAttention(name=f"part_{i+1}")(region))

fused = layers.Concatenate(name="feature_concat")(features)
x = layers.Dense(EMBEDDING_SIZE, name="embedding_linear")(fused)
x = layers.BatchNormalization(name="embedding_bn")(x)
outputs = layers.Dense(NUM_IDS, activation="softmax", name="classifier")(x)
model = keras.Model(inputs, outputs, name="PAT_inspired")
embedding_model = keras.Model(inputs, layers.UnitNormalization(axis=-1)(x))
model.summary()
print(model(tf.zeros((1, IMAGE_SIZE, IMAGE_SIZE, 3)), training=False).shape)

## Retrieval metrics
Validation stays as one dataset. Evaluate all-vs-all, excluding the identical image. Test uses the saved query/gallery files. AP@100 divides by min(number of relevant images,100); verify against the official evaluator before comparing Kaggle scores.

In [ ]:
def retrieval_metrics(q, g, q_ids, g_ids, same_set=False):
    q_ids, g_ids = np.asarray(q_ids), np.asarray(g_ids)
    scores = q @ g.T
    ap, ranks = [], []
    for i, row in enumerate(scores):
        valid = np.ones(len(g), dtype=bool)
        if same_set:
            valid[i] = False
        relevant = (g_ids == q_ids[i]) & valid
        total = relevant.sum()
        if total == 0:
            continue
        indices = np.flatnonzero(valid)
        order = indices[np.argsort(-row[indices], kind="stable")[:100]]
        hits = relevant[order]
        precision = np.cumsum(hits) / np.arange(1, len(hits)+1)
        ap.append(np.sum(precision * hits) / min(total, 100))
        ranks.append([hits[:k].any() for k in (1, 5, 10)])
    if not ap:
        raise ValueError("No valid queries")
    r = np.mean(ranks, axis=0)
    return {"mAP@100": float(np.mean(ap)), "Rank-1": float(r[0]),
            "Rank-5": float(r[1]), "Rank-10": float(r[2]), "valid_queries": len(ap)}

class ValidationRetrieval(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        z = embedding_model.predict(val_ds, verbose=0)
        metrics = retrieval_metrics(z, z, val.object_id, val.object_id, same_set=True)
        logs["val_map"] = metrics["mAP@100"]
        print("\nValidation:", metrics)

CHECKPOINT = OUT / "pat_best.weights.h5"
class BestRetrieval(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = -1.0
    def on_epoch_end(self, epoch, logs=None):
        if logs["val_map"] > self.best:
            self.best = logs["val_map"]
            self.model.save_weights(CHECKPOINT)

best = BestRetrieval()
def fit_stage(lr, epochs):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model.fit(train_ds, epochs=epochs, callbacks=[ValidationRetrieval(), best,
        keras.callbacks.EarlyStopping(monitor="val_map", mode="max", patience=4)])

history_head = fit_stage(1e-3, HEAD_EPOCHS)
model.load_weights(CHECKPOINT)

## Fine-tuning
Unfreeze the backbone at a low learning rate. The backbone is called with training=False to disable stochastic training behavior; gradients still update trainable weights. Reduce batch size if GPU memory is insufficient. Best validation checkpoint is shared across both stages.

In [ ]:
backbone.trainable = True
history_fine = fit_stage(1e-5, FINE_TUNE_EPOCHS)
model.load_weights(CHECKPOINT)
embedding_model.save(OUT / "pat_embedding.keras")

## Final local test
Run only after selecting the final model. No camera filtering, no augmentation.

In [ ]:
q = embedding_model.predict(query_ds, verbose=1)
g = embedding_model.predict(gallery_ds, verbose=1)
results = retrieval_metrics(q, g, query.object_id, gallery.object_id)
print(results)
pd.DataFrame([results]).to_csv(OUT / "pat_test_metrics.csv", index=False)
np.save(OUT / "pat_query_embeddings.npy", q)
np.save(OUT / "pat_gallery_embeddings.npy", g)
order = np.argsort(-(q @ g.T), axis=1, kind="stable")[:, :100]
pd.DataFrame(gallery.image_path.to_numpy()[order]).to_csv(OUT / "pat_local_ranking.csv", index=False)